In [90]:
import geopandas as gpd
import pandas as pd
from functools import reduce 
from functions import *

Estimates relative to Residential Housing for the second semester (S2) of 2025 are merged into geographical perimeters.

Load OMI estimates

In [113]:
df_omi = pd.read_csv('datasets/omi_estimate/omi_estimate.csv')
add_zeroes(df_omi, 'mun_istat', 6)

,year,year_semester,semester,zone,type,condition,buy_min,buy_max,mun_istat,mun_name,prov_name,reg_name
0,2014,2014_S1,1,B1,Residential housing,Normal,680,960,006003,Alessandria,Alessandria,Piemonte
1,2014,2014_S1,1,B1,Garages,Normal,1100,1600,006003,Alessandria,Alessandria,Piemonte
2,2014,2014_S1,1,B1,Covered parking spaces,Normal,700,1000,006003,Alessandria,Alessandria,Piemonte
3,2014,2014_S1,1,B1,Uncovered parking spaces,Normal,600,800,006003,Alessandria,Alessandria,Piemonte
4,2014,2014_S1,1,B1,Warehouses,Normal,1050,1300,006003,Alessandria,Alessandria,Piemonte
...,...,...,...,...,...,...,...,...,...,...,...,...
3823677,2025,2025_S2,2,B1,Warehouses,Normal,300,350,113019,Padru,Gallura Nord-Est Sardegna,Sardegna
3823678,2025,2025_S2,2,B1,Shops,Normal,550,750,113019,Padru,Gallura Nord-Est Sardegna,Sardegna
3823679,2025,2025_S2,2,B1,Offices,Normal,520,710,113019,Padru,Gallura Nord-Est Sardegna,Sardegna
3823680,2025,2025_S2,2,B1,Industrial buildings,Normal,360,450,113019,Padru,Gallura Nord-Est Sardegna,Sardegna


In [114]:
# Select 2025_S2 estimate for Residential Housing
df_omi = df_omi[(df_omi['year_semester'] == '2025_S2') & (df_omi['type'] == 'Residential housing')]

Zone data

In [117]:
# Load zone perimeters
gdf_zone = gpd.read_file('datasets/geo_data/omi_zone_perimeters.gpkg')

In [118]:
# Merge geographical info into estimates
gdf_zone_merged = gpd.GeoDataFrame(pd.merge(
    df_omi, gdf_zone[['mun_istat', 'zone', 'geometry']], 
    on = ['mun_istat', 'zone'], 
    how = 'left'
))

gdf_zone_merged = gdf_zone_merged.dropna(subset = 'geometry')

In [119]:
gdf_zone_merged.to_file('datasets/data_maps/zone_estimate.gpkg')

Municipal data

In [120]:
# Load Municipality perimeters
gdf_mun = gpd.read_file('datasets/geo_data/mun_perimeters.gpkg')

c:\Users\HP\Desktop\projects\data_municipalities\.venv_website_housing\Lib\site-packages\pyogrio\geopandas.py:382: UserWarning: More than one layer found in 'mun_perimeters.gpkg': 'municipalities' (default), 'mun_perimeters'. Specify layer parameter to avoid this warning.
  result = read_func(


In [123]:
# Group by ISTAT code
df_omi_mun = df_omi.groupby('mun_istat').aggregate({
    'mun_name' : 'first',
    'prov_name' : 'first',
    'reg_name' : 'first',
    'year' : 'first',
    'year_semester' : 'first',
    'buy_min' : 'mean',
    'buy_max' : 'mean'
}).reset_index()

# Change buy_min/max to int (no decimals)
df_omi_mun[['buy_min', 'buy_max']] = df_omi_mun[['buy_min', 'buy_max']].astype(int)

# Merge geographical info into estimates
gdf_mun_merged = gpd.GeoDataFrame(pd.merge(
    df_omi_mun, gdf_mun[['mun_istat', 'geometry']], 
    on = 'mun_istat', 
    how = 'left'
))

gdf_mun_merged = gdf_mun_merged.dropna()

In [125]:
gdf_mun_merged.to_file('datasets/data_maps/mun_estimate.gpkg')

Province data

In [126]:
# Load Province perimeters
gdf_prov = gpd.read_file('datasets/geo_data/prov_perimeters.gpkg')

In [127]:
# Group by Province names
df_omi_prov = df_omi.groupby('prov_name').aggregate({
    'reg_name' : 'first',
    'year' : 'first',
    'year_semester' : 'first',
    'buy_min' : 'mean',
    'buy_max' : 'mean'
}).reset_index()

# Change buy_min/max to int (no decimals)
df_omi_prov[['buy_min', 'buy_max']] = df_omi_prov[['buy_min', 'buy_max']].astype(int)

# Merge geographical info into estimates
gdf_prov_merged = gpd.GeoDataFrame(pd.merge(
    df_omi_prov, gdf_prov[['prov_name', 'geometry']],
    on = 'prov_name',
    how = 'left'
))

In [128]:
gdf_prov_merged.to_file('datasets/data_maps/prov_estimate.gpkg')

Region data

In [129]:
# Load Region perimeters
gdf_reg = gpd.read_file('datasets/geo_data/reg_perimeters.gpkg')

In [130]:
# Group by Region names
df_omi_reg = df_omi.groupby('reg_name').aggregate({
    'year' : 'first',
    'year_semester' : 'first',
    'buy_min' : 'mean',
    'buy_max' : 'mean'
}).reset_index()

# Change buy_min/max to int (no decimals)
df_omi_reg[['buy_min', 'buy_max']] = df_omi_reg[['buy_min', 'buy_max']].astype(int)

# Merge geographical info into estimates
gdf_reg_merged = gpd.GeoDataFrame(pd.merge(
    df_omi_reg, gdf_reg,
    on = 'reg_name',
    how = 'left'
))

In [131]:
gdf_reg_merged.to_file('datasets/data_maps/reg_estimate.gpkg')